# Reproduce the ValuePrism pipeline

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JeffVallyath/geometry-of-endorsement/blob/main/notebooks/01_reproduce_valueprism_pipeline.ipynb)

The main dataset is ValuePrism. Each example gives us a situation, an action, and a moral reason. The dataset records whether that reason Supports or Opposes the action.

This notebook rebuilds the split and checkerboard inputs. The training and test sets share no situations and no exact reason labels.

`DEMO` checks the saved results. `FULL` rebuilds them from ValuePrism and saves the output to Google Drive.

## Run instructions

1. Use `DEMO` for the quick check.
2. For `FULL`, accept the ValuePrism license and add `HF_TOKEN` to Colab Secrets.
3. Choose Runtime, then Run all.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys

RUN_MODE = "DEMO"
VALID_MODES = ('DEMO', 'FULL')
PUBLIC_REPOSITORY = 'https://github.com/JeffVallyath/geometry-of-endorsement.git'
SOURCE_COMMIT = '0ddaada4dc7595d0c4d46651b7d81b8a251b0658'

if RUN_MODE not in VALID_MODES:
    raise ValueError(f"RUN_MODE must be one of {VALID_MODES}")

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_ROOT = Path("/content/geometry-of-endorsement-reproduction")
    if not (REPO_ROOT / ".git").is_dir():
        if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
            raise RuntimeError("The checkout directory is not empty")
        REPO_ROOT.mkdir(parents=True, exist_ok=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "init"], check=True)
        subprocess.run(["git", "-C", str(REPO_ROOT), "remote", "add", "origin", PUBLIC_REPOSITORY], check=True)
    origin = subprocess.run(
        ["git", "-C", str(REPO_ROOT), "remote", "get-url", "origin"],
        check=True, text=True, capture_output=True,
    ).stdout.strip()
    if origin.rstrip("/").removesuffix(".git") != PUBLIC_REPOSITORY.rstrip("/").removesuffix(".git"):
        raise RuntimeError("The checkout has an unexpected origin")
    dirty = subprocess.run(
        ["git", "-C", str(REPO_ROOT), "status", "--porcelain"],
        check=True, text=True, capture_output=True,
    ).stdout
    if dirty:
        raise RuntimeError("The checkout contains local changes")
    subprocess.run(["git", "-C", str(REPO_ROOT), "fetch", "--depth", "1", "origin", SOURCE_COMMIT], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "checkout", "--detach", SOURCE_COMMIT], check=True)
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    REPO_ROOT = next(path for path in candidates if (path / "pyproject.toml").is_file())

head = subprocess.run(
    ["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"],
    check=True, text=True, capture_output=True,
).stdout.strip()
if IN_COLAB and head != SOURCE_COMMIT:
    raise RuntimeError(f"Expected source commit {SOURCE_COMMIT}, found {head}")


subprocess.run([sys.executable, "-m", "pip", "install", "-q", str(REPO_ROOT), "--no-deps"], check=True)
SOURCE_ROOT = str(REPO_ROOT / "src")
if SOURCE_ROOT not in sys.path:
    sys.path.insert(0, SOURCE_ROOT)

print({"mode": RUN_MODE, "source_commit": SOURCE_COMMIT, "repo_root": str(REPO_ROOT)})

In [ ]:
from IPython.display import display

from geometry_of_truth.leakage.contracts import load_bundle
from geometry_of_truth.leakage.results import audit, overlap_checks, stress_draws

bundle = load_bundle(REPO_ROOT)
print("Retained aggregate integrity verified")
display(audit(bundle["results"]))
display(overlap_checks(bundle["results"]))
display(stress_draws(bundle["results"]))

In [ ]:
if RUN_MODE == "FULL":
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"{REPO_ROOT}[valueprism-full]"],
        check=True,
    )
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    token = userdata.get("HF_TOKEN")
    if not token:
        raise RuntimeError("Add HF_TOKEN to Colab Secrets")
    os.environ["HF_TOKEN"] = token
    OUTPUT_ROOT = Path("/content/drive/MyDrive/geometry-of-endorsement/valueprism-reproduction")
    from geometry_of_truth.leakage.reproduce import reproduce

    run = reproduce(OUTPUT_ROOT)
    display(run["comparison"])
    if not bool(run["comparison"]["pass"].all()):
        raise RuntimeError("The ValuePrism reconstruction differs from the retained aggregate")
else:
    OUTPUT_ROOT = None
    print("Set RUN_MODE to FULL to rebuild the licensed manifests")

In [ ]:
metadata = {
    "experiment_id": "2026-08-11-valueprism-pipeline",
    "hypothesis": "Withheld consideration identities create a larger leakage penalty than withheld situations alone.",
    "dataset": "allenai/ValuePrism",
    "dataset_revision": "d439ca90825e5b4e5ef97798d9b5950e16ba7065",
    "mode": RUN_MODE,
    "source_commit": SOURCE_COMMIT,
    "output_root": str(OUTPUT_ROOT) if OUTPUT_ROOT else None,
}
print(json.dumps(metadata, indent=2, sort_keys=True))
if OUTPUT_ROOT:
    (OUTPUT_ROOT / "run_metadata.json").write_text(
        json.dumps(metadata, indent=2, sort_keys=True) + "\n", encoding="utf-8"
    )

## Reading the result

`FULL` must match every saved count, hash, overlap check, and five-seed result. Use its output for the Llama notebook.